# Calculate the percent decrease of the growth rates between the nude and immunocompetent mice

## Load required packages

In [3]:
%matplotlib widget
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from src import pyxfunc

## Setup

In [5]:
growth_df = pd.read_csv("data/growth_data_df.csv")
exclude = [456, 458, 461, 464, 471, 474, 478, 482, 483, 426, 428, 429, 430, 434, 437, 438, 442, 443, 451, 642, 662]
growth_df = growth_df[~(growth_df["id"].isin(exclude))]
gs =  np.arange(0.01, 0.25, 0.01)

## Functions

In [10]:
def run_exponential(g, c, end_time, subline):
    if subline == 0: # C1
        return pyxfunc.run(c, 0, 0, 7, end_time, 6, np.inf, 0.001, 0.01, 0, g, 0, 0, 0, 0, 0, 0, 0, 1)
    else: # C11
        return pyxfunc.run(0, c, 0, 7, end_time, 6, np.inf, 0.001, 0.01, 0, 0, g, 0, 0, 0, 0, 0, 0, 1)

In [15]:
def get_growth_rate(group, mid, gs):
    errs = []
    idxs = [(d-7)*100 for d in growth_df[growth_df["id"] == mid]["day"].tolist()]
    sizes = growth_df[growth_df["id"] == mid]["size"].to_numpy()
    sizes = sizes/sizes[0]
    start_size = growth_df[(growth_df["id"]==mid) & (growth_df["day"]==7)]["size"].tolist()[0]
    
    for g in gs:
        sol = run_exponential(g, start_size, growth_df[growth_df["id"]==mid]["day"].max(), group)
        sol[0] = np.asarray(sol[0])/start_size
        sol[1] = np.asarray(sol[1])/start_size
        err = pyxfunc.get_error(idxs, sizes, sol, group, 0.02)
        errs += [err[0]]
    best_g = gs[np.argmin(errs)]
    return best_g

## C1

In [22]:
c1_nude_grs = []
for mid_nude in growth_df[growth_df["group"]=="Grp. B1 nude (100% C1)"]["id"].unique():
    nude_gr = get_growth_rate(0, mid_nude, gs)
    c1_nude_grs += [nude_gr]
c1_nude_mean = np.mean(c1_nude_grs)

c1_b6_grs = []
for mid_b6 in growth_df[growth_df["group"]=="Grp. A1 B6 (100% C1)"]["id"].unique():
    b6_gr = get_growth_rate(0, mid_b6, gs)
    c1_b6_grs += [b6_gr]
c1_b6_mean = np.mean(c1_b6_grs)

print(c1_nude_mean)
print(c1_b6_mean)

0.18500000000000003
0.053333333333333344


In [23]:
c1_b6_mean / c1_nude_mean

np.float64(0.2882882882882883)

## C11

In [25]:
c11_nude_grs = []
for mid_nude in growth_df[growth_df["group"]=="Grp. B5 nude (100% C11)"]["id"].unique():
    nude_gr = get_growth_rate(0, mid_nude, gs)
    c11_nude_grs += [nude_gr]
c11_nude_mean = np.mean(c11_nude_grs)

c11_b6_grs = []
for mid_b6 in growth_df[growth_df["group"]=="Grp. A5 B6 (100% C11)"]["id"].unique():
    b6_gr = get_growth_rate(0, mid_b6, gs)
    c11_b6_grs += [b6_gr]
c11_b6_mean = np.mean(c11_b6_grs)

print(c11_nude_mean)
print(c11_b6_mean)

0.1366666666666667
0.10600000000000001


In [26]:
c11_b6_mean / c11_nude_mean

np.float64(0.775609756097561)